# 🚗 Spark Optimization Masterclass
### Using the Car Sales Star Schema Dataset (`SalesData_cars.csv`)

---

| # | Topic |
|---|-------|
| 1 | Imports & Raw Schema |
| 2 | Spark Session & All Key Configurations |
| 3 | AutoLoader — What it is & How it works |
| 4 | AutoLoader — Running the Stream |
| 5 | Incremental Deduplication (Anti-Join) |
| 6 | Data Cleaning Pipeline |
| 7 | Partitioning — Write |
| 8 | Partitioning — Partition Pruning Demo |
| 9 | Bucketing — Write |
| 10 | Bucketing — Join Without Shuffle |
| 11 | Broadcast Join — Explicit `F.broadcast()` |
| 12 | Broadcast Join — Auto Threshold |
| 13 | Sort-Merge Join — Shuffle Explained |
| 14 | Shuffle Partitions — Impact Demo |
| 15 | Caching — `cache()` vs No Cache |
| 16 | Persistence — `persist()` Storage Levels |
| 17 | Predicate Pushdown — Filter Early |
| 18 | Column Pruning — Read Less I/O |
| 19 | Z-Ordering — Delta Data Skipping |
| 20 | Adaptive Query Execution (AQE) |
| 21 | Skew Detection & Salting Fix |
| 22 | Reading Query Plans (`explain()`) |
| 23 | Final Star Query — All Optimizations Together |
| 24 | Best Practices Summary & Cleanup |


---
## 📦 Cell 1 — Imports & Raw Schema

We define the schema **upfront** instead of letting Spark infer it.

**Why?**
- Schema inference requires Spark to read the entire file once just to guess types → wasted scan
- On large files or many files, inference is slow and can be wrong (e.g., a Revenue column with NULLs inferred as `string`)
- Explicit schema = faster, predictable, production-safe

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType
)
from pyspark.sql.window import Window
from pyspark import StorageLevel
from delta.tables import DeltaTable
import time

# ── Define schema upfront (never let Spark infer in production)
raw_schema = StructType([
    StructField("Branch_ID",    StringType(),  True),
    StructField("Dealer_ID",    StringType(),  True),
    StructField("Model_ID",     StringType(),  True),
    StructField("Revenue",      LongType(),    True),
    StructField("Units_Sold",   IntegerType(), True),
    StructField("Date_ID",      StringType(),  True),
    StructField("Day",          IntegerType(), True),
    StructField("Month",        IntegerType(), True),
    StructField("Year",         IntegerType(), True),
    StructField("BranchName",   StringType(),  True),
    StructField("DealerName",   StringType(),  True),
    StructField("Product_Name", StringType(),  True),
])

print(f"✅ Schema defined — {len(raw_schema.fields)} columns")
for f in raw_schema.fields:
    print(f"   {f.name:<18} → {f.dataType}")

---
## ⚙️ Cell 2 — Spark Session & All Key Configurations

The `SparkSession` is the **single entry point** to everything in Spark — reading data, running SQL, configuring the engine. One per application.

| Config | What it does | Our value | When to change |
|--------|-------------|-----------|----------------|
| `spark.sql.shuffle.partitions` | Partitions created after a shuffle (join/groupBy). Default=200 | `8` | ~ data_MB / 128, min 8 |
| `spark.sql.autoBroadcastJoinThreshold` | Auto-broadcast tables smaller than this. `-1` disables | `10MB` | Increase for bigger dims |
| `spark.sql.adaptive.enabled` | AQE: re-plans query at runtime using real stats | `true` | Keep always true |
| `spark.sql.adaptive.coalescePartitions.enabled` | AQE merges tiny post-shuffle partitions | `true` | Keep always true |
| `spark.sql.adaptive.skewJoin.enabled` | AQE splits hot (skewed) partitions during joins | `true` | Keep always true |
| `spark.databricks.delta.optimizeWrite.enabled` | Delta compacts small files on every write | `true` | Keep true in Databricks |
| `spark.sql.files.maxPartitionBytes` | Max bytes per partition when reading files | `128MB` | Lower for more parallelism |

In [0]:
spark = (
    SparkSession.builder
    .appName("SparkOptimizationMasterclass")

    # ── Delta Lake extensions
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    # ── Shuffle partition tuning
    # Rule of thumb: (total data size in MB / 128) or (num_cores x 2~4)
    # Our dataset is ~1800 rows so 8 is plenty.
    # Production 100GB+ data → use 400-800
    .config("spark.sql.shuffle.partitions", "8")

    # ── Broadcast join threshold: auto-broadcast tables < 10MB
    .config("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

    # ── Adaptive Query Execution: re-optimize at runtime
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")

    # ── Delta write optimization: merges small files on write
    .config("spark.databricks.delta.optimizeWrite.enabled", "true")

    # ── Max bytes Spark reads into one partition from file storage
    .config("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))

    .getOrCreate()
)

print("✅ Spark session ready — version:", spark.version)
print()
configs = [
    "spark.sql.shuffle.partitions",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.databricks.delta.optimizeWrite.enabled",
    "spark.sql.files.maxPartitionBytes",
]
for c in configs:
    print(f"   {c.split('.')[-1]:<40} = {spark.conf.get(c)}")

---
## 📥 Cell 3 — AutoLoader: What It Is & How It Works

AutoLoader (`cloudFiles`) is Databricks' **incremental file ingestion** engine.

**The problem it solves:**
Without AutoLoader, you'd need to manually track which files were already loaded. Every time a new CSV drops in your cloud folder, you'd have to diff against a list and only load new ones. That's fragile.

**How AutoLoader works:**
```
  Cloud Storage folder
  ├── SalesData_cars.csv        ← Run 1: picked up ✅, checkpoint saved
  ├── IncrementalSales_Jan.csv  ← Run 2: NEW → picked up ✅
  └── IncrementalSales_Feb.csv  ← Run 3: NEW → picked up ✅

  Checkpoint folder (NEVER DELETE)
  └── Remembers: { SalesData_cars.csv: ✅, IncrementalSales_Jan.csv: ✅ ... }
```

**Key options explained:**

| Option | Purpose |
|--------|---------|
| `cloudFiles.format` | File format: `csv`, `json`, `parquet`, `avro` |
| `cloudFiles.schemaLocation` | Where AutoLoader writes its inferred/saved schema. Required. |
| `cloudFiles.schemaEvolutionMode` | `rescue` = store unknown columns in `_rescued_data`. `addNewColumns` = auto-add. |
| `checkpointLocation` | Tracks processed files. **Never delete this folder.** |
| `trigger(once=True)` | Process all pending files then stop (batch mode). |
| `toTable(...)` | Write to a managed Delta table (ACID, queryable, audit-ready). |

---
## 🚀 Cell 4 — AutoLoader: Running the Stream

- `readStream` opens a streaming reader — AutoLoader watches the folder
- `writeStream ... trigger(once=True)` runs it as a **one-shot batch** — ideal for scheduled pipelines
- `awaitTermination()` blocks until the stream finishes (all pending files processed)
- On the **second run**, the checkpoint says "SalesData_cars.csv already done" and skips it automatically

In [0]:
print("▶ Starting AutoLoader ingestion…")

(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("encoding", "UTF-8")
    # rescue: unknown columns stored in _rescued_data instead of failing
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    # AutoLoader saves its inferred schema here so it doesn't re-infer every run
    .option("cloudFiles.schemaLocation",
            "/Volumes/workspace/fact_to_dim&fact/source_files/schema/")
    .schema(raw_schema)
    .load("/Volumes/workspace/fact_to_dim&fact/source_files")
    .writeStream
    # Checkpoint = AutoLoader's memory. Records which files are done.
    .option("checkpointLocation",
            "/Volumes/workspace/fact_to_dim&fact/source_files/checkpoint/")
    .option("mergeSchema", "true")
    # trigger(once=True): process all pending files, then stop.
    # Use trigger(availableNow=True) on newer Databricks runtimes.
    .trigger(once=True)
    # Write to a Delta table — gives us ACID, time-travel, and easy querying
    .toTable("workspace.`fact_to_dim&fact`.fact_sales_staging")
    .awaitTermination()
)

print("✅ AutoLoader run complete — staging table updated")

df_staging = spark.table("workspace.`fact_to_dim&fact`.fact_sales_staging")
print(f"   Staging rows      : {df_staging.count():,}")
print(f"   Staging partitions: {df_staging.rdd.getNumPartitions()}")

---
## 🔁 Cell 5 — Incremental Deduplication: The Anti-Join

**The problem:** AutoLoader writes to `fact_sales_staging` cumulatively.
On Run 2, staging contains old rows + new rows. If we process everything,
we duplicate data already in `fact_sales`.

**The fix: `left_anti` join**
```
  left_anti join keeps ONLY rows in LEFT table (staging)
  that have NO match in RIGHT table (fact_sales)

  staging  →  [row_A ✅ in fact_sales]  ← dropped
              [row_B ✅ in fact_sales]  ← dropped
              [row_C ❌ not in fact_sales] ← KEPT  ← new!
              [row_D ❌ not in fact_sales] ← KEPT  ← new!
```
This is cleaner than timestamp-based filtering because it's based on actual content,
not metadata that could be missing or unreliable.

In [0]:
try:
    # Keys already committed to the gold fact table
    existing_keys = (
        spark.table("workspace.`fact_to_dim&fact`.fact_sales")
        .select("Branch_ID", "Dealer_ID", "Model_ID",
                "Date_ID", "Revenue", "Units_Sold")
    )

    # left_anti: keep ONLY staging rows with no match in existing_keys
    df_raw = df_staging.join(
        existing_keys,
        on=["Branch_ID", "Dealer_ID", "Model_ID",
            "Date_ID", "Revenue", "Units_Sold"],
        how="left_anti"
    )
    print(f"✅ Incremental mode")
    print(f"   Total staging rows : {df_staging.count():,}")
    print(f"   Already processed  : {existing_keys.count():,}")
    print(f"   NEW rows to process: {df_raw.count():,}")

except Exception:
    # First run — fact_sales does not exist yet
    df_raw = df_staging
    print(f"✅ First load mode — all {df_raw.count():,} rows are new")

---
## 🧹 Cell 6 — Data Cleaning Pipeline

Three-step cleaning on the new rows only:
1. **`dropna`** on key columns — rows without Branch/Dealer/Model/Date IDs are useless
2. **`trim`** all string columns — CSV files often carry leading/trailing spaces that break joins
3. **`dropDuplicates`** — removes exact duplicate rows from the new batch

Then we **`.cache()`** the result because every cell below reuses `df_clean`.
Without caching, Spark would re-read staging and redo all cleaning on every action.

In [0]:
string_cols = ["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID",
               "BranchName", "DealerName", "Product_Name"]

# Step 1: drop rows with null key columns
df_clean = df_raw.dropna(
    subset=["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID"]
)

# Step 2: trim whitespace from all string columns
for col in string_cols:
    df_clean = df_clean.withColumn(col, F.trim(F.col(col)))

# Step 3: remove exact duplicate rows
df_clean = df_clean.dropDuplicates()

# Cache — we'll reuse df_clean in every section below
# Without cache: Spark re-reads staging + re-cleans on every .count() / .show()
df_clean.cache()
total = df_clean.count()  # triggers cache materialisation

print("─── NULL CHECK ──────────────────────────────")
for col in ["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID", "Revenue"]:
    n = df_clean.filter(F.col(col).isNull()).count()
    print(f"   {col:<18} → {n} nulls")

print(f"\n✅ Clean rows cached: {total:,}")
print(f"   Partitions in memory: {df_clean.rdd.getNumPartitions()}")

---
## 📂 Cell 7 — Partitioning: Write a Partitioned Delta Table

Partitioning physically splits your data into **subfolders** based on a column value:
```
fact_sales_partitioned/
├── Year=2017/
│   ├── Month=1/  → only Jan-2017 rows stored here
│   ├── Month=2/
│   └── ...
├── Year=2018/
├── Year=2019/
└── Year=2020/
```

**Rules of thumb:**
- ✅ Partition on **low-cardinality** columns you **frequently filter on** (Year, Month, Country)
- ❌ Never partition on high-cardinality columns like `Dealer_ID` → 250 folders = metadata explosion
- ❌ Never partition on columns with many NULLs → a `Year=null/` folder is useless
- Aim for **≥ 128 MB per partition file** to avoid the small-files problem

In [0]:
# Add a Full_Date column for richer downstream use
df_fact_partitioned = df_clean.withColumn(
    "Full_Date",
    F.to_date(
        F.concat_ws("-", F.col("Year"), F.col("Month"), F.col("Day")),
        "yyyy-M-d"
    )
)

print("Writing partitioned Delta table (Year / Month)…")

(
    df_fact_partitioned.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("Year", "Month")    # ← creates Year=.../Month=... folder hierarchy
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.fact_sales_partitioned")
)

print("✅ Partitioned table written")

# Show the partition layout metadata
spark.sql("""
    DESCRIBE DETAIL workspace.`fact_to_dim&fact`.fact_sales_partitioned
""").select("numFiles", "sizeInBytes", "partitionColumns").show(truncate=False)

# List distinct partitions
spark.sql("""
    SHOW PARTITIONS workspace.`fact_to_dim&fact`.fact_sales_partitioned
""").show(10)

---
## ✂️ Cell 8 — Partitioning: Partition Pruning Demo

**Partition pruning** = Spark reads ONLY the folder(s) that match your WHERE clause.
All other folders are completely skipped — zero I/O.

```
  WHERE Year=2019 AND Month=3
       ↓
  Reads: fact_sales_partitioned/Year=2019/Month=3/  ✅
  Skips: Year=2017/, Year=2018/, Year=2020/ ...     ❌ (never touched)
```

Without partitioning, both queries below would read the entire table.

In [0]:
# ── Query 1: NO partition filter — full table scan
print("─── FULL SCAN (no partition filter) ───")
spark.sql("""
    SELECT Year, SUM(Revenue) as Total_Revenue, COUNT(*) as Num_Sales
    FROM   workspace.`fact_to_dim&fact`.fact_sales_partitioned
    GROUP  BY Year
    ORDER  BY Year
""").show()

# ── Query 2: WITH partition filter — only Year=2019/Month=3 folder read
print("─── PARTITION PRUNING (Year=2019, Month=3 only) ───")
spark.sql("""
    SELECT Year, Month, SUM(Revenue) as Total_Revenue,
           SUM(Units_Sold) as Total_Units, COUNT(*) as Transactions
    FROM   workspace.`fact_to_dim&fact`.fact_sales_partitioned
    WHERE  Year  = 2019
    AND    Month = 3
    GROUP  BY Year, Month
""").show()

# ── Check the plan — look for 'PartitionFilters' near the bottom Scan node
print("\n── Query plan (look for PartitionFilters):")
spark.table("workspace.`fact_to_dim&fact`.fact_sales_partitioned") \
     .filter((F.col("Year") == 2019) & (F.col("Month") == 3)) \
     .explain(mode="simple")

---
## 🪣 Cell 9 — Bucketing: Write Bucketed Tables

Bucketing goes a step beyond partitioning. Inside each file, rows are
**pre-hashed and pre-sorted** into N fixed buckets based on a join key:

```
  bucketBy(8, "Dealer_ID"):
  bucket 0 → rows where hash(Dealer_ID) % 8 == 0
  bucket 1 → rows where hash(Dealer_ID) % 8 == 1
  ...
  bucket 7 → rows where hash(Dealer_ID) % 8 == 7
```

**Why bucket?** When you join two tables bucketed on the SAME column with the SAME N,
Spark knows bucket 0 of table A can ONLY join with bucket 0 of table B.
The shuffle is completely eliminated.

| | Partitioning | Bucketing |
|--|--|--|
| Eliminates | Full table scans (WHERE filters) | Shuffle during joins |
| Column type | Low cardinality | Medium/high cardinality join key |
| Storage | Subfolders | Fixed N files |
| Requires | Nothing extra | Hive metastore (`saveAsTable`) |

> ⚠️ Both tables **must** use the same column AND the same bucket count.

In [0]:
# Build the dimension table to bucket against
df_dim_dealer = (
    df_clean
    .select("Dealer_ID", "DealerName")
    .dropDuplicates(["Dealer_ID"])
    .withColumn("DimDealer_SK", F.monotonically_increasing_id() + 1)
)
print(f"Distinct dealers: {df_dim_dealer.count()}")

# ── Write FACT table: 8 buckets on Dealer_ID
print("\nWriting bucketed fact table…")
(
    df_clean.write
    .format("parquet")
    .mode("overwrite")
    .bucketBy(8, "Dealer_ID")   # ← 8 buckets, keyed on Dealer_ID
    .sortBy("Dealer_ID")        # ← pre-sort within each bucket
    .option("path",
            "/Volumes/workspace/fact_to_dim&fact/gold/fact_sales_bucketed")
    .saveAsTable(
        "workspace.`fact_to_dim&fact`.fact_sales_bucketed"
    )
)

# ── Write DIM table: SAME bucket config (column + count must match!)
print("Writing bucketed dim_dealer…")
(
    df_dim_dealer.write
    .format("parquet")
    .mode("overwrite")
    .bucketBy(8, "Dealer_ID")   # ← must match: same column, same N
    .sortBy("Dealer_ID")
    .option("path",
            "/Volumes/workspace/fact_to_dim&fact/gold/dim_dealer_bucketed")
    .saveAsTable(
        "workspace.`fact_to_dim&fact`.dim_dealer_bucketed"
    )
)

print("✅ Both bucketed tables written")

---
## 🔗 Cell 10 — Bucketing: Join Without Shuffle

Now we join the two bucketed tables. Spark detects they share the same
bucket config and **skips the Exchange (shuffle) step entirely**.

In the query plan you should see:
- `SortMergeJoin` ← the join operator
- **No `Exchange hashpartitioning`** on either side ← this is the proof the shuffle is gone

We disable broadcast first so Spark is forced to use the bucketed join path.

In [0]:
# Disable broadcast to force Spark to use the bucketed join path
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

df_bucketed_join = (
    spark.table("workspace.`fact_to_dim&fact`.fact_sales_bucketed")
    .join(
        spark.table("workspace.`fact_to_dim&fact`.dim_dealer_bucketed"),
        on="Dealer_ID",
        how="inner"
    )
)

print("── Join plan — look for SortMergeJoin WITHOUT Exchange nodes:")
df_bucketed_join.explain(mode="simple")

print(f"\n✅ Bucketed join rows: {df_bucketed_join.count():,}")
print("   No shuffle occurred — data was already co-located by bucket!")

# Restore broadcast threshold for remaining cells
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

---
## 📡 Cell 11 — Broadcast Join: Explicit `F.broadcast()`

In a standard join, Spark shuffles BOTH tables so matching keys land on the same partition.
In a **Broadcast Join**, the small table is serialised and sent to EVERY executor in full.
Each executor joins its local slice of the big table against its local copy of the small one — **zero shuffle**.

```
  Normal join:                     Broadcast join:
  ─────────────────────────────    ──────────────────────────────────────
  Big Table  ──shuffle──►          Big Table  (stays where it is)
  Small Table──shuffle──►    +     Small Table ──broadcast──► all executors
              join                             join  (local, no shuffle)
```

**When to use:** one table is clearly small (dimension tables, lookup tables < 50 MB)

**When NOT to use:** both tables are large — broadcasting a 2 GB table will OOM your executors

In [0]:
# Build our small dimension tables
df_dim_branch = (
    df_clean.select("Branch_ID", "BranchName")
    .dropDuplicates(["Branch_ID"])
)
df_dim_product = (
    df_clean.select("Model_ID", "Product_Name")
    .dropDuplicates(["Model_ID"])
)
df_dim_date = (
    df_clean.select("Date_ID", "Day", "Month", "Year")
    .dropDuplicates(["Date_ID"])
    .withColumn("Full_Date",
        F.to_date(F.concat_ws("-", "Year", "Month", "Day"), "yyyy-M-d"))
    .withColumn("Quarter", F.quarter("Full_Date"))
)

print("── Table sizes:")
print(f"   fact (df_clean)  : {df_clean.count():>6,} rows  ← LARGE — never broadcast")
print(f"   dim_branch       : {df_dim_branch.count():>6,} rows  ← small — great for broadcast")
print(f"   dim_product      : {df_dim_product.count():>6,} rows  ← small — great for broadcast")
print(f"   dim_date         : {df_dim_date.count():>6,} rows  ← small — great for broadcast")

print("\n── EXPLICIT BROADCAST JOIN using F.broadcast():")
# F.broadcast() is a hint — tells Spark to broadcast this specific DataFrame
df_with_branch = df_clean.join(
    F.broadcast(df_dim_branch),   # ← explicit broadcast hint
    on="Branch_ID",
    how="left"
)

# In the plan: look for BroadcastHashJoin and BroadcastExchange
# You will NOT see Exchange hashpartitioning on the big table side
df_with_branch.explain(mode="simple")
print(f"\nRows: {df_with_branch.count():,}")

---
## 🤖 Cell 12 — Broadcast Join: Auto Threshold

Instead of manually wrapping every small table with `F.broadcast()`,
you can set `autoBroadcastJoinThreshold` and Spark will automatically
broadcast any table smaller than that byte limit.

All three dimension joins below will be auto-broadcast:

In [0]:
# Threshold already set to 10MB in the SparkSession config
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(f"autoBroadcastJoinThreshold = {int(threshold) // (1024*1024)} MB")
print("All three dim tables are well under 10MB → auto-broadcast will trigger\n")

# All three joins will be BroadcastHashJoin — no explicit F.broadcast() needed
df_star = (
    df_clean
    .join(df_dim_branch,  on="Branch_ID",  how="left")  # auto-broadcast
    .join(df_dim_product, on="Model_ID",   how="left")  # auto-broadcast
    .join(df_dim_date,    on="Date_ID",    how="left")  # auto-broadcast
    .select("BranchName", "Product_Name", "Quarter",
            "Year", "Revenue", "Units_Sold")
)

print("── AUTO BROADCAST plan (all three dims broadcast):")
df_star.explain(mode="simple")

print(f"\nFull star join rows: {df_star.count():,}")

---
## 🔀 Cell 13 — Sort-Merge Join: Shuffle Explained

When broadcast is disabled (both tables too large), Spark falls back to **Sort-Merge Join (SMJ)**:

```
  Step 1 — SHUFFLE:  Both tables redistributed over the network.
                     Rows with the same join key land on the same partition.
  Step 2 — SORT:     Each partition sorted by the join key.
  Step 3 — MERGE:    Sorted partitions merged like a zipper — O(n) scan.
```

In the query plan you will see:
- `Exchange hashpartitioning(Branch_ID, 8)` ← the shuffle step
- `Sort [Branch_ID ASC]` ← the sort step  
- `SortMergeJoin` ← the actual merge

This is why `shuffle.partitions` is so important — it controls the `8` in `hashpartitioning(key, 8)`.

In [0]:
# Disable broadcast to force Sort-Merge Join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("─── SORT-MERGE JOIN (broadcast disabled) ───")
df_smj = (
    df_clean
    .join(df_dim_branch,  on="Branch_ID",  how="left")
    .join(df_dim_product, on="Model_ID",   how="left")
)

# Look for Exchange hashpartitioning (the shuffle) + SortMergeJoin
df_smj.explain(mode="simple")
print(f"\nRows: {df_smj.count():,}")

# Restore broadcast threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
print("\n✅ Broadcast threshold restored to 10MB")

---
## 🔢 Cell 14 — Shuffle Partitions: Impact Demo

`spark.sql.shuffle.partitions` controls how many output partitions are created
after every shuffle (groupBy, join, window functions).

**Default = 200. This is almost always wrong:**

- Small data (1800 rows) + 200 partitions = 200 tasks each processing ~9 rows. Massive overhead.
- Large data (500GB) + 200 partitions = each partition is 2.5GB. OOM.

**Formula:**
```
  shuffle.partitions = max(8,  total_shuffle_data_MB / 128)
  or
  shuffle.partitions = num_executor_cores * 2~4
```

In [0]:
# ── Test with DEFAULT 200 partitions (bad for small data)
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")  # disable AQE to see raw result

df_200 = df_clean.groupBy("Year", "Month").agg(F.sum("Revenue"))
df_200.count()   # trigger
print(f"shuffle.partitions=200 → output partitions: {df_200.rdd.getNumPartitions()}")
print("  Each partition has only a handful of rows — huge task scheduling overhead!")

# ── Test with TUNED 8 partitions (right for our dataset)
spark.conf.set("spark.sql.shuffle.partitions", "8")

df_8 = df_clean.groupBy("Year", "Month").agg(F.sum("Revenue"))
df_8.count()   # trigger
print(f"\nshuffle.partitions=8  → output partitions: {df_8.rdd.getNumPartitions()}")
print("  8 balanced partitions — right-sized for our data volume.")

# Re-enable AQE coalescing
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Show the actual aggregation
print("\n── Revenue by Year and Month:")
df_8.withColumnRenamed("sum(Revenue)", "Total_Revenue") \
    .orderBy("Year", "Month").show()

---
## 💾 Cell 15 — Caching: `cache()` vs No Cache

Spark is **lazy and stateless by default**. Every action re-executes the entire DAG from source.
If you call `.count()` three times on the same DataFrame, Spark reads the file three times.

`.cache()` stores the result in **executor memory** after first computation.
Subsequent actions read from RAM — much faster.

```
  Without cache:    read → clean → join  ← action 1
                    read → clean → join  ← action 2  (repeated!)
                    read → clean → join  ← action 3  (repeated!)

  With cache:       read → clean → join  ← action 1 (+ stores in memory)
                    memory read           ← action 2  ✅ fast
                    memory read           ← action 3  ✅ fast
```

In [0]:
df_heavy = (
    df_clean
    .join(F.broadcast(df_dim_branch),  on="Branch_ID",  how="left")
    .join(F.broadcast(df_dim_product), on="Model_ID",   how="left")
    .join(F.broadcast(df_dim_date),    on="Date_ID",    how="left")
    .withColumn("Revenue_Per_Unit",
                (F.col("Revenue") / F.col("Units_Sold")).cast("long"))
)

# ── WITHOUT cache: Spark re-reads + re-joins for every single action
print("─── WITHOUT CACHE ───")
t0 = time.time()
_ = df_heavy.filter(F.col("Year") == 2019).count()              # full DAG run #1
_ = df_heavy.groupBy("Product_Name").agg(F.sum("Revenue")).count()  # full DAG run #2
_ = df_heavy.filter(F.col("Quarter") == 2).count()              # full DAG run #3
t_no_cache = time.time() - t0
print(f"   3 actions elapsed: {t_no_cache:.2f}s")

# ── WITH cache: compute once, serve all three actions from memory
print("\n─── WITH CACHE ───")
df_heavy.cache()
df_heavy.count()   # warm up the cache

t0 = time.time()
_ = df_heavy.filter(F.col("Year") == 2019).count()              # from memory ✅
_ = df_heavy.groupBy("Product_Name").agg(F.sum("Revenue")).count()  # from memory ✅
_ = df_heavy.filter(F.col("Quarter") == 2).count()              # from memory ✅
t_cache = time.time() - t0
print(f"   3 actions elapsed: {t_cache:.2f}s")

print(f"\n⚡ Cache speedup: {t_no_cache/max(t_cache,0.001):.1f}x faster")

# IMPORTANT: Always unpersist when you are done with the DataFrame!
# Keeping stale caches wastes executor memory and can cause OOM.
df_heavy.unpersist()
print("\n✅ df_heavy unpersisted — memory freed")

---
## 🗄️ Cell 16 — Persistence: `persist()` Storage Levels

`.cache()` is just `.persist(MEMORY_AND_DISK)` with a default level.
`persist()` lets you choose the storage level explicitly:

| Storage Level | Where | Serialized? | Recompute on eviction? | Use when |
|---|---|---|---|---|
| `MEMORY_ONLY` | RAM | No | ✅ Yes (expensive) | Small DFs, JVM objects |
| `MEMORY_AND_DISK` | RAM → spill to disk | No | ❌ No | Default choice |
| `MEMORY_ONLY_SER` | RAM | ✅ Yes (compact) | ✅ Yes | Large DFs, tight on RAM |
| `MEMORY_AND_DISK_SER` | RAM → disk | ✅ Yes | ❌ No | Large DFs, safest option |
| `DISK_ONLY` | Disk | ✅ Yes | ❌ No | Rarely needed |
| `OFF_HEAP` | Off-heap RAM | ✅ Yes | ❌ No | Avoid GC pressure |

**Rule of thumb:** Use `MEMORY_AND_DISK` for most workloads.

In [0]:
from pyspark import StorageLevel

# MEMORY_AND_DISK: RAM first, spill to disk if executor runs out of memory
# This is the safest choice for production pipelines
df_clean.persist(StorageLevel.MEMORY_AND_DISK)
df_clean.count()   # trigger persistence
print(f"✅ df_clean persisted to MEMORY_AND_DISK")
print(f"   Rows available: {df_clean.count():,}")

# Check what's currently cached in Spark
print("\n── Cached RDDs (visible in Spark UI → Storage tab):")
for rdd in spark.sparkContext._jsc.sc().getRDDStorageInfo():
    print(f"   {rdd.name()} — {rdd.storageLevel()}")

# Unpersist — always clean up to free executor memory
df_clean.unpersist()

# Re-cache at default level for remaining cells
df_clean.cache()
df_clean.count()
print("\n✅ df_clean re-cached at default (MEMORY_AND_DISK) for remaining cells")

---
## 🔍 Cell 17 — Predicate Pushdown: Filter Early

**Predicate pushdown** = push the WHERE clause as close to the data source as possible.

Delta Lake / Parquet supports pushdown, meaning Spark can ask the storage layer:
*"I only want rows where Year=2019"* — and the storage returns only those rows,
never loading the rest into memory.

```
  ❌ BAD: join all data first, then filter
     [read ALL years] → [join ALL years] → [filter Year=2019]
     Spark processed 1800 rows through the join unnecessarily.

  ✅ GOOD: filter first, then join the smaller result
     [read Year=2019 folder only] → [join ~400 rows]
     4x less data going into the join.
```

In [0]:
# ❌ BAD: join first, filter after — Spark reads and joins ALL years
df_bad = (
    spark.table("workspace.`fact_to_dim&fact`.fact_sales_partitioned")
    .join(F.broadcast(df_dim_branch), on="Branch_ID", how="left")
    .filter(F.col("Year") == 2019)   # filter comes AFTER the join
)

# ✅ GOOD: filter on partitioned column FIRST — partition pruning kicks in
# Then join only the ~400 filtered rows
df_good = (
    spark.table("workspace.`fact_to_dim&fact`.fact_sales_partitioned")
    .filter(F.col("Year") == 2019)   # partition pruning here — reads only Year=2019/
    .join(F.broadcast(df_dim_branch), on="Branch_ID", how="left")
)

print("❌ BAD plan — filter after join:")
df_bad.explain(mode="simple")

print("\n✅ GOOD plan — filter before join (note PartitionFilters near Scan):")
df_good.explain(mode="simple")

print(f"Both return: {df_good.count():,} rows (same result, very different cost)")

---
## ✂️ Cell 18 — Column Pruning: Read Less I/O

Parquet (and Delta) is a **columnar format**. Each column is stored in its own set of files.
If your table has 12 columns but your query only needs 4, Spark reads only those 4 column files.

```
  12-column table:
  Branch_ID │ Dealer_ID │ Model_ID │ Revenue │ Units_Sold │ Date_ID │ ... (6 more)

  SELECT Branch_ID, Model_ID, Revenue, Units_Sold → reads 4/12 column files = 67% less I/O
  SELECT *                                         → reads all 12 column files
```

**Always SELECT only the columns you need. Never use `SELECT *` in production.**

In [0]:
tbl = spark.table("workspace.`fact_to_dim&fact`.fact_sales_partitioned")

# ❌ BAD: reads all 13 column files from Parquet
df_all = tbl.select("*")
print(f"❌ SELECT * reads  : {len(df_all.columns)} column files")

# ✅ GOOD: Parquet only opens 6 out of 13 column files
df_pruned = tbl.select(
    "Branch_ID", "Model_ID", "Year", "Month", "Revenue", "Units_Sold"
)
print(f"✅ SELECT 6 reads  : {len(df_pruned.columns)} column files  "
      f"({100*(1 - len(df_pruned.columns)/len(df_all.columns)):.0f}% less I/O)")

# Check the query plan — look for 'ReadSchema' at the bottom
# The pruned plan will list only the 6 columns in ReadSchema
print("\n── Pruned plan (ReadSchema shows only 6 columns):")
df_pruned.filter(F.col("Year") == 2019).explain(mode="simple")

# Practical demo: aggregate on pruned columns
print("── Revenue summary using pruned read:")
df_pruned.groupBy("Year") \
    .agg(
        F.sum("Revenue").alias("Total_Revenue"),
        F.sum("Units_Sold").alias("Total_Units")
    ).orderBy("Year").show()

---
## ⚡ Cell 19 — Z-Ordering: Delta Data Skipping

Z-Ordering is a **multi-dimensional data clustering** technique exclusive to Delta Lake.
It physically reorganises data within Delta files so rows with similar values on the
Z-ordered column(s) are stored together.

Delta tracks **min/max statistics per file** for each column. After Z-ordering:

```
  Before Z-ORDER on BranchName:
    File 1: BranchName min="AC Cars"    max="Zonda"    → can't skip on ANY filter
    File 2: BranchName min="Acura"      max="Zwit"     → can't skip on ANY filter

  After OPTIMIZE ZORDER BY (BranchName):
    File 1: BranchName min="AC Cars"    max="Anadol"   → skip if filter > "Anadol"
    File 2: BranchName min="Ariel"      max="BMW"      → skip if filter outside range
    File 3: BranchName min="Cadillac"   max="Ferrari"  → skip if filter outside range
```

| | Partition | Z-Order |
|--|--|--|
| Column cardinality | Low (Year, Month) | Medium/high (BranchName, Product) |
| How it works | Folder-level skip | File-level skip (min/max stats) |
| When applied | At write time | Via `OPTIMIZE` command |
| Overhead | Write time | Separate OPTIMIZE step |

In [0]:
# ── Run OPTIMIZE + ZORDER
# This rewrites Delta files clustering rows by BranchName and Product_Name
print("Running OPTIMIZE + ZORDER BY (BranchName, Product_Name)…")
spark.sql("""
    OPTIMIZE workspace.`fact_to_dim&fact`.fact_sales_partitioned
    ZORDER BY (BranchName, Product_Name)
""")
print("✅ ZORDER complete")

# ── Query that benefits from Z-ordering (BranchName + Product_Name filter)
print("\n── Filtered query (data skipping will skip non-matching files):")
spark.sql("""
    SELECT BranchName, Product_Name,
           SUM(Revenue)    as Total_Revenue,
           SUM(Units_Sold) as Total_Units,
           COUNT(*)        as Transactions
    FROM   workspace.`fact_to_dim&fact`.fact_sales_partitioned
    WHERE  BranchName   = 'Ariel Motors'
    AND    Product_Name = 'Honda'
    GROUP  BY BranchName, Product_Name
""").show(truncate=False)

# ── Delta history shows the OPTIMIZE operation and how many files were compacted
print("── OPTIMIZE operation in Delta history:")
spark.sql("""
    DESCRIBE HISTORY workspace.`fact_to_dim&fact`.fact_sales_partitioned
""").filter(F.col("operation") == "OPTIMIZE") \
   .select("version", "operation", "operationMetrics") \
   .show(3, truncate=False)

---
## 🤖 Cell 20 — Adaptive Query Execution (AQE)

Classic Spark plans queries **before execution** using estimated statistics.
Estimates are often wrong — especially after filters or skewed joins.

**AQE re-plans at runtime** using actual row counts and data sizes from completed stages.

### AQE's 3 superpowers:

**1. Dynamic Partition Coalescing**
After a shuffle with 200 partitions, AQE merges tiny/empty ones automatically.
No more 200 tasks each processing 9 rows.

**2. Dynamic Join Strategy Switch**
If AQE discovers at runtime that one join side is tiny after a filter,
it switches from SortMergeJoin to BroadcastHashJoin mid-execution.

**3. Dynamic Skew Join Optimization**
If one partition has 10× more rows than others, AQE splits the hot partition
and duplicates the matching partition from the other table side.

In [0]:
print("── AQE settings:")
aqe_configs = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize",
]
for c in aqe_configs:
    try:
        print(f"   {c.split('.')[-1]:<45} = {spark.conf.get(c)}")
    except Exception:
        print(f"   {c.split('.')[-1]:<45} = (default)")

# ── Demo 1: AQE partition coalescing
# Start with 200 shuffle partitions — AQE will merge the empty/tiny ones
print("\n─── AQE COALESCING DEMO ───")
spark.conf.set("spark.sql.shuffle.partitions", "200")
# AQE coalesce is already enabled
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

df_aqe = df_clean.groupBy("Year", "Month").agg(
    F.sum("Revenue").alias("Total_Revenue"),
    F.count("*").alias("Num_Sales")
)
df_aqe.show()
actual_parts = df_aqe.rdd.getNumPartitions()
print(f"   Logical shuffle.partitions : 200")
print(f"   Actual output partitions   : {actual_parts}")
print(f"   AQE merged {200 - actual_parts} empty partitions automatically!")

# Restore
spark.conf.set("spark.sql.shuffle.partitions", "8")

---
## ⚖️ Cell 21 — Skew Detection & The Salting Fix

**Data skew** = some partitions hold far more rows than others.

```
  Executor 1: partition with BranchName='Ariel' → 800,000 rows  ← HOT
  Executor 2: partition with BranchName='BMW'   →   5,000 rows
  Executor 3: partition with BranchName='Ford'  →   4,800 rows

  Job finishes only when ALL executors finish.
  → Executor 2 & 3 sit idle waiting for Executor 1 (the straggler).
```

**Salting fix:** Add a random integer (0..N-1) to the hot key to split it into N sub-keys.
The matching side is **exploded** to carry all N salt variants.

```
  Before salting:  Branch='Ariel'           ← all 800k rows in one partition
  After salting:   Branch='Ariel_0' (200k)  ← spread across 4 partitions
                   Branch='Ariel_1' (200k)
                   Branch='Ariel_2' (200k)
                   Branch='Ariel_3' (200k)
```

In [0]:
# ── Step 1: Detect skew — which branches have the most rows?
print("─── SKEW DETECTION: rows per Branch_ID ───")
df_skew = (
    df_clean
    .groupBy("Branch_ID", "BranchName")
    .agg(F.count("*").alias("row_count"))
    .orderBy(F.desc("row_count"))
)
df_skew.show(10)

max_rows = df_skew.agg(F.max("row_count")).collect()[0][0]
avg_rows = df_skew.agg(F.avg("row_count")).collect()[0][0]
print(f"Max rows in one branch: {max_rows}")
print(f"Avg rows per branch   : {avg_rows:.1f}")
if max_rows > avg_rows * 5:
    print("⚠️  Skew detected — max is >5x the average")
else:
    print("✅ No significant skew in this dataset (good!)")

# ── Step 2: Salting technique (demonstrated even if skew is mild)
NUM_SALTS = 4
print(f"\n─── SALTING WITH {NUM_SALTS} BUCKETS ───")

# Add a random salt 0..3 to each fact row
df_salted_fact = df_clean.withColumn(
    "salted_key",
    F.concat(
        F.col("Branch_ID"),
        F.lit("_"),
        (F.rand() * NUM_SALTS).cast("int")
    )
)

# Explode the dimension so every Branch_ID carries all 4 salt variants
df_salted_branch = (
    df_dim_branch
    .withColumn("salt_explode", F.array([F.lit(i) for i in range(NUM_SALTS)]))
    .withColumn("salt", F.explode("salt_explode"))
    .withColumn("salted_key",
                F.concat(F.col("Branch_ID"), F.lit("_"), F.col("salt").cast("string")))
    .drop("salt_explode", "salt")
)

# Join on the salted key — hot partition is now spread across 4 sub-partitions
df_anti_skew = (
    df_salted_fact
    .join(F.broadcast(df_salted_branch), on="salted_key", how="left")
    .drop("salted_key")
)

print(f"✅ Salted join rows: {df_anti_skew.count():,}")
print(f"   Hot partition is now spread across {NUM_SALTS} sub-partitions")
print("   No single executor is a straggler anymore")

---
## 🔬 Cell 22 — Reading Query Plans with `explain()`

**Always read query plans bottom to top** — bottom is where data is read, top is the final output.

| Node | Meaning |
|---|---|
| `FileScan` / `Scan` | Reading data from disk |
| `Filter` | WHERE clause being applied |
| `PartitionFilters` | Partition pruning (only relevant folders read) |
| `Exchange hashpartitioning` | ⚠️ Shuffle happening — expensive network I/O |
| `BroadcastExchange` | Broadcast happening — cheap, no shuffle on big side |
| `Sort` | Data being sorted (usually before SortMergeJoin) |
| `SortMergeJoin` | Large-table join (after shuffle + sort) |
| `BroadcastHashJoin` | ✅ Broadcast join — no shuffle on big table |
| `HashAggregate` | groupBy / aggregation |
| `Project` | SELECT — column selection/computation |

**Three explain modes:**
- `explain(mode="simple")` — compact, most useful for daily use
- `explain(mode="extended")` — shows parsed + analyzed + optimized + physical plan
- `explain(mode="formatted")` — cleanest structured output

In [0]:
df_explain_demo = (
    spark.table("workspace.`fact_to_dim&fact`.fact_sales_partitioned")
    .filter((F.col("Year") == 2019) & (F.col("Month").isin([1, 2, 3])))  # partition pruning
    .select("Branch_ID", "Model_ID", "Revenue", "Units_Sold", "Year", "Month")  # column pruning
    .join(F.broadcast(df_dim_branch),  on="Branch_ID",  how="left")   # broadcast join
    .join(F.broadcast(df_dim_product), on="Model_ID",   how="left")   # broadcast join
    .groupBy("BranchName", "Product_Name")
    .agg(F.sum("Revenue").alias("Total_Revenue"),
         F.count("*").alias("Txns"))
    .orderBy(F.desc("Total_Revenue"))
)

print("══ SIMPLE PLAN (read bottom → top) ══")
print("Bottom: FileScan → PartitionFilters (Year=2019, Month IN [1,2,3])")
print("Middle: BroadcastHashJoin (no Exchange = no shuffle on fact side)")
print("Top:    HashAggregate + Sort")
print()
df_explain_demo.explain(mode="simple")

print("\n══ FORMATTED PLAN (most readable) ══")
df_explain_demo.explain(mode="formatted")

---
## 🏆 Cell 23 — Final Star Query: All Optimizations Together

This single query applies every technique from this masterclass simultaneously:

| Optimization | Where applied |
|---|---|
| Partition pruning | `WHERE Year IN (2019, 2020)` on partitioned table |
| Column pruning | `.select(6 columns)` instead of `SELECT *` |
| Broadcast join | All three dimension joins use `F.broadcast()` |
| Predicate pushdown | Filter applied before joins |
| Cache | `df_clean` already in memory from Cell 6 |
| Z-Ordering | `fact_sales_partitioned` already Z-ordered on BranchName |
| AQE | Active — will coalesce partitions and optimize joins at runtime |
| shuffle.partitions=8 | Right-sized for our data volume |

In [0]:
print("═" * 60)
print("  FINAL OPTIMIZED STAR QUERY")
print("  Active: partition pruning + column pruning + broadcast")
print("          + predicate pushdown + AQE + Z-order")
print("═" * 60)

df_final = (
    spark.table("workspace.`fact_to_dim&fact`.fact_sales_partitioned")

    # ── Partition pruning: skip Year=2017 and Year=2018 folders entirely
    .filter(F.col("Year").isin([2019, 2020]))

    # ── Column pruning: Parquet reads only these 7 column files
    .select("Branch_ID", "Model_ID", "Date_ID",
            "Revenue", "Units_Sold", "Year", "Month")

    # ── Broadcast joins: all dims < 10MB → zero shuffle on fact side
    .join(F.broadcast(df_dim_branch),  on="Branch_ID",  how="left")
    .join(F.broadcast(df_dim_product), on="Model_ID",   how="left")
    .join(F.broadcast(df_dim_date),    on="Date_ID",    how="left")

    # ── Aggregation
    .groupBy("BranchName", "Product_Name", "Quarter", "Year")
    .agg(
        F.sum("Revenue").alias("Total_Revenue"),
        F.sum("Units_Sold").alias("Total_Units"),
        F.count("*").alias("Transactions"),
        F.round(F.avg("Revenue"), 0).cast("long").alias("Avg_Revenue_Per_Sale")
    )
    .orderBy(F.desc("Total_Revenue"))
)

df_final.show(20, truncate=False)
print(f"\nTotal result rows : {df_final.count():,}")

print("\n── Optimized query plan:")
df_final.explain(mode="simple")

---
## 📋 Cell 24 — Best Practices Summary & Cleanup

| Topic | Rule |
|---|---|
| **Schema** | Always define schema explicitly. Never use `inferSchema` in production. |
| **AutoLoader** | Never delete the checkpoint folder — it's AutoLoader's memory. |
| **shuffle.partitions** | Tune to `data_MB / 128`. Default 200 is wrong for most workloads. |
| **Partitioning** | Low-cardinality columns (Year, Month, Region). Avoid high-cardinality. |
| **Bucketing** | Same column + same N on both joined tables. Needs `saveAsTable`. |
| **Broadcast Join** | Tables < 10–50MB. Use `F.broadcast()` or tune `autoBroadcastJoinThreshold`. |
| **Sort-Merge Join** | Default for large tables. Ensure shuffle.partitions is tuned. |
| **Caching** | Only cache DataFrames used 2+ times. Always `unpersist()` when done. |
| **Predicate Pushdown** | Filter EARLY, before joins. Use partition columns in WHERE first. |
| **Column Pruning** | Always `.select()` only needed columns. Never `SELECT *` in prod. |
| **Z-Ordering** | For medium/high cardinality filter columns. Run `OPTIMIZE` after large loads. |
| **AQE** | Keep enabled always. It's free optimization — leaves Databricks default on. |
| **Skew** | Detect with `groupBy(key).count()`. Fix with AQE skewJoin or salting. |
| **explain()** | Read bottom-up. Spot `Exchange` (shuffles) and push filters down to `Scan`. |

In [0]:
# ── Final cleanup: release all cached data to free executor memory
df_clean.unpersist()
spark.catalog.clearCache()

print("═" * 60)
print("  🎓 SPARK OPTIMIZATION MASTERCLASS — COMPLETE")
print("═" * 60)
print()
topics = [
    "✅ Imports & Raw Schema",
    "✅ Spark Session & Key Configurations",
    "✅ AutoLoader (cloudFiles)",
    "✅ Incremental Deduplication (Anti-Join)",
    "✅ Data Cleaning Pipeline",
    "✅ Partitioning (write + partition pruning)",
    "✅ Bucketing (write + shuffle-free join)",
    "✅ Broadcast Join (explicit + auto threshold)",
    "✅ Sort-Merge Join & Shuffle mechanics",
    "✅ Shuffle Partitions impact demo",
    "✅ Caching (cache vs no-cache timing)",
    "✅ Persistence (storage levels)",
    "✅ Predicate Pushdown",
    "✅ Column Pruning",
    "✅ Z-Ordering & Delta data skipping",
    "✅ Adaptive Query Execution (AQE)",
    "✅ Skew Detection & Salting",
    "✅ Reading Query Plans (explain)",
    "✅ Final Star Query (all optimizations)",
]
for t in topics:
    print(f"   {t}")

print()
print("All caches cleared. Cluster memory freed.")